In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report
import os

# 1. Cargar datos buscando en varias rutas
rutas = ['../worldbank_pib.csv', '../data/worldbank_pib.csv', '../notebooks/worldbank_pib.csv']
for r in rutas:
    if os.path.exists(r):
        df = pd.read_csv(r)
        break

# 2. CREAR LA VARIABLE BINARIA (1 o 0)
df['crecimiento_alto'] = (df['crecimiento_pib_pct'] >= 3.0).astype(int)

# 3. Seleccionar variables
features = ['pib_per_capita_usd', 'inflacion_pct', 'desempleo_pct']
target = 'crecimiento_alto'
df_ml = df.dropna(subset=features + [target]).copy()

X = df_ml[features]
y = df_ml[target]

# 4. Dividir datos (NO es necesario escalar para árboles)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Entrenar Árbol de Clasificación
modelo_arbol_clas = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
modelo_arbol_clas.fit(X_train, y_train)

# 6. Evaluar
y_pred = modelo_arbol_clas.predict(X_test)
print("=== Árbol de Clasificación Binaria ===")
print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred):.4f}")
print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred))

# Mostrar las reglas del árbol
print("\nReglas del Árbol:")
print(export_text(modelo_arbol_clas, feature_names=features))

=== Árbol de Clasificación Binaria ===
Exactitud (Accuracy): 0.5667

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.50      0.62      0.55        13
           1       0.64      0.53      0.58        17

    accuracy                           0.57        30
   macro avg       0.57      0.57      0.57        30
weighted avg       0.58      0.57      0.57        30


Reglas del Árbol:
|--- pib_per_capita_usd <= 44459.79
|   |--- pib_per_capita_usd <= 5161.54
|   |   |--- pib_per_capita_usd <= 2037.43
|   |   |   |--- class: 0
|   |   |--- pib_per_capita_usd >  2037.43
|   |   |   |--- desempleo_pct <= 17.12
|   |   |   |   |--- class: 1
|   |   |   |--- desempleo_pct >  17.12
|   |   |   |   |--- class: 0
|   |--- pib_per_capita_usd >  5161.54
|   |   |--- inflacion_pct <= 3.65
|   |   |   |--- desempleo_pct <= 10.61
|   |   |   |   |--- class: 1
|   |   |   |--- desempleo_pct >  10.61
|   |   |   |   |--- class: 0
|   |   |--- infl